# 03.04 章节实践：独立完成双缓冲调度

## 小节概述

本节是一项独立编程实践。你将修改工作副本中的 <code>student_pipeline.h</code>：把单缓冲基线改成两个物理 Buffer，并完成预装、预取、上一结果写回和最终排空。

<strong>完成标志：</strong> Host MockPipeline 在 <code>tileCount=1、2、7</code> 下证明调度正确，Ascend C 工程干净构建，四类非法参数正确拒绝，四组 NPU 全量精度用例通过，末尾显示 <code>SELF_CHECK PASS</code>。

> starter 是可运行且数值正确的单缓冲基线，但不满足本章的双缓冲调度要求。开始修改后，不要重新运行未修改的 starter 单元，否则会覆盖你的代码。


## 1. 实践任务

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>项目</th><th style='text-align: left;'>要求</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'>TODO 1</td><td style='text-align: left;'>把 <code>kStudentBufferNum</code> 从 1 改为 2</td></tr>
    <tr><td style='text-align: left;'>TODO 2</td><td style='text-align: left;'>循环前预装 Tile 0；每轮 DeQue 当前、预取下一 Tile、写回上一结果、计算当前；循环后排空最后结果</td></tr>
    <tr><td style='text-align: left;'>允许修改</td><td style='text-align: left;'>工作副本中的 <code>student_pipeline.h</code></td></tr>
    <tr><td style='text-align: left;'>不应修改</td><td style='text-align: left;'><code>vector_add_pipeline_student.asc</code>、CMake、评分器、MockPipeline 和 JSON 用例</td></tr>
    <tr><td style='text-align: left;'>调度不变量</td><td style='text-align: left;'>每个 Tile 的 CopyIn/Compute/CopyOut 各一次；<code>tileCount</code> 不除以 2；无越界、无无符号下溢、最终队列全部排空</td></tr>
  </tbody>
</table>


## 2. 准备独立工作副本

运行下一单元后，学生工程、评分器和用例会复制到当前系统用户专属且可复用的临时工作目录。不同用户不再共享同一个 <code>/tmp</code> 子目录。默认继续已有工作，避免覆盖已完成的 TODO；确需重置时，把 <code>RESET_WORKSPACE</code> 临时改为 <code>True</code>。


In [ ]:
from pathlib import Path
import getpass
import os
import re
import shutil
import subprocess
import sys
import tempfile

RESET_WORKSPACE = False
USER_KEY = f'uid_{os.getuid()}' if hasattr(os, 'getuid') else f'user_{getpass.getuser()}'
USER_TEMP_ROOT = Path(tempfile.gettempdir()) / f'cannlab_data_structures_compute_{USER_KEY}'
WORK_ROOT = USER_TEMP_ROOT / '03_double_buffer_pipeline' / 'chapter_test'
previous_repo = globals().get('REPO_ROOT')
try:
    current = Path.cwd().resolve()
except FileNotFoundError:
    cached_repo = Path(previous_repo) if previous_repo is not None else None
    if cached_repo is None or not (cached_repo / 'contrib/tutorials/data_structures_compute').is_dir():
        raise RuntimeError(
            '当前内核的工作目录已被清理，请重启内核后从本节第一个代码单元开始运行'
        ) from None
    os.chdir(cached_repo)
    current = cached_repo.resolve()

def outside_work_root(path):
    try:
        Path(path).resolve().relative_to(WORK_ROOT)
        return False
    except ValueError:
        return True

search_roots = []
if previous_repo is not None:
    search_roots.append(Path(previous_repo).resolve())
search_roots.extend([current, *current.parents])
REPO_ROOT = next(
    (p for p in search_roots if outside_work_root(p) and (p / 'contrib/tutorials/data_structures_compute').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('请从 cann-learning-hub 仓库内打开本 Notebook')
os.chdir(REPO_ROOT)
CHAPTER = REPO_ROOT / 'contrib/tutorials/data_structures_compute/03_double_buffer_pipeline'
if RESET_WORKSPACE and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def copy_missing_tree(source, target):
    target.mkdir(parents=True, exist_ok=True)
    for source_path in source.rglob('*'):
        target_path = target / source_path.relative_to(source)
        if source_path.is_dir():
            target_path.mkdir(parents=True, exist_ok=True)
        elif not target_path.exists():
            target_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, target_path)

copy_missing_tree(CHAPTER / 'src/practice', WORK_ROOT / 'practice')
copy_missing_tree(CHAPTER / 'src/tests', WORK_ROOT / 'tests')
if not (WORK_ROOT / 'check_double_buffer.py').is_file():
    shutil.copy2(CHAPTER / 'src/grader/grade_double_buffer.py', WORK_ROOT / 'check_double_buffer.py')
PROJECT = WORK_ROOT / 'practice'
CHECKER = WORK_ROOT / 'check_double_buffer.py'
ANSWER = CHAPTER / 'answer/03.04_chapter_test/student_pipeline.h'
STUDENT_HEADER_FILE = PROJECT / 'student_pipeline.h'
print('student project:', PROJECT)
subprocess.run(['ls', '-R', str(WORK_ROOT)], check=True)


## 3. 查看关键源码与接口

学生头文件只控制物理 Buffer 数和 Process 调度；完整 Tensor、队列、DataCopy 与 Add 实现已经位于学生工程中。运行下一单元，用 <code>cat</code> 查看 starter，并定位五个可调用接口。


In [ ]:
subprocess.run(['cat', str(PROJECT / 'student_pipeline.h')], check=True)
print('\n--- Pipeline interface locations ---')
source_lines = (PROJECT / 'vector_add_pipeline_student.asc').read_text(encoding='utf-8').splitlines()
for index, line in enumerate(source_lines, start=1):
    if any(name in line for name in ['void CopyIn(', 'DeQueX()', 'DeQueY()', 'void Compute(', 'void CopyOut(']):
        print(f'{index:>3}: {line}')


## 4. 写入 starter

初始化单元已准备当前用户的学生实践文件，下一单元使用 <code>%%writefile {STUDENT_HEADER_FILE}</code> 通过变量绝对路径写入可运行的单缓冲基线，不会切换 Notebook 工作目录。第一次运行后，在该单元中补全 TODO 并重新运行。注意：每次运行都会覆盖工作副本中的头文件。


In [ ]:
%%writefile {STUDENT_HEADER_FILE}
#pragma once

#include <cstdint>

// TODO 1：双缓冲需要为每个 TQue 分配两个物理 Buffer。
constexpr int32_t kStudentBufferNum = 1;

// TODO 2：把严格串行调度改为预装、预取、上一结果写回和最终排空。
template <typename Pipeline>
__aicore__ inline void StudentProcess(Pipeline &pipeline, uint32_t tileCount)
{
    for (uint32_t tile = 0; tile < tileCount; ++tile) {
        pipeline.CopyIn(tile);
        auto xLocal = pipeline.DeQueX();
        auto yLocal = pipeline.DeQueY();
        pipeline.Compute(xLocal, yLocal);
        pipeline.CopyOut(tile);
    }
}


## 5. 实现顺序

建议按下面顺序修改：

1. 把 <code>kStudentBufferNum</code> 改为 2；
2. 在循环前添加 <code>CopyIn(0)</code>；
3. 循环内先 DeQue 当前输入；
4. 使用 <code>tile + 1 &lt; tileCount</code> 判断下一块；
5. 使用 <code>tile &gt; 0</code> 后再写回 <code>tile - 1</code>，防止 unsigned 下溢；
6. 计算当前 Tile；
7. 循环后写回 <code>tileCount - 1</code>。

不要先连续 CopyIn 两块再 DeQue：queue depth 固定为 1。也不要把循环次数改成 <code>tileCount/2</code>。

![双缓冲预取与排空时序](./images/double_buffer_timeline.svg)


In [ ]:
header = PROJECT / 'student_pipeline.h'
text = header.read_text(encoding='utf-8')
quick_checks = [
    ('物理 Buffer 数已改为 2', bool(re.search(r'kStudentBufferNum\s*=\s*2\s*;', text))),
    ('存在循环前预装', 'CopyIn(0)' in text),
    ('存在下一 Tile 边界', 'tile + 1 < tileCount' in text),
    ('存在上一 Tile 边界', 'tile > 0' in text),
    ('存在最终排空', 'CopyOut(tileCount - 1)' in text),
]
for description, passed in quick_checks:
    print(f"[{'PASS' if passed else 'CHECK'}] {description}")
print('快速检查不证明调用顺序或 NPU 数值正确，最终以完整自检为准。')


## 6. 运行完整自检

完整自检分四层：

1. 用普通 Host C++ 编译器实例化你的模板，在 MockPipeline 上检查 <code>tileCount=1、2、7</code> 的调用顺序、槽位占用和最终排空；
2. 在干净目录中配置并编译 Ascend C 学生工程；
3. 运行整除、对齐和 UB 超限四类非法参数；
4. 在 NPU 上运行四组合法 Shape，逐元素比较 CPU Golden，并核对 <code>buffer_num=2</code>、<code>queue_depth=1</code> 和 <code>schedule=prefetch</code>。

评分器不会要求双缓冲达到固定加速比，因为 Host 计时噪声不适合作为功能实验的硬门槛。


In [ ]:
build_dir = WORK_ROOT / 'build'
check_result = subprocess.run(
    [
        sys.executable, str(CHECKER), '--project', str(PROJECT),
        '--build-dir', str(build_dir),
        '--cases', str(WORK_ROOT / 'tests/pipeline_cases.json'),
        '--schedule-test', str(WORK_ROOT / 'tests/pipeline_schedule_test.cpp'),
    ],
    text=True, capture_output=True, check=False,
)
raw = (check_result.stdout or '') + ('\n[stderr]\n' + check_result.stderr if check_result.stderr else '')
visible = [line for line in raw.splitlines() if not line.startswith('AUTO_RESULT_')]
print('\n'.join(visible))
print('self-check exit code:', check_result.returncode)
print('SELF_CHECK PASS' if check_result.returncode == 0 else 'SELF_CHECK FAIL')
first_failure = next((line for line in visible if '[FAIL]' in line), None)
if check_result.returncode != 0 and first_failure:
    print('请先处理最早的失败项：', first_failure)


## 7. 常见失败定位

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>现象</th><th style='text-align: left;'>优先检查</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>kStudentBufferNum must be 2</code></td><td style='text-align: left;'>TODO 1 是否仍为 1</td></tr>
    <tr><td style='text-align: left;'><code>no observable input prefetch window</code></td><td style='text-align: left;'>是否在当前 DeQue 后、当前 Compute 前预取下一 Tile</td></tr>
    <tr><td style='text-align: left;'><code>CopyOut reads an empty queue</code></td><td style='text-align: left;'>是否在 Tile 0 错写 <code>tile-1</code>，或在 Compute 前写当前 Tile</td></tr>
    <tr><td style='text-align: left;'><code>output queue was not drained</code></td><td style='text-align: left;'>循环后是否遗漏最后一次 CopyOut</td></tr>
    <tr><td style='text-align: left;'>clean Ascend C build 失败</td><td style='text-align: left;'>从第一条编译错误检查模板语法、分号与接口参数</td></tr>
    <tr><td style='text-align: left;'>某个 precision case 失败</td><td style='text-align: left;'>是否跳过 Tile、重复 Tile，或提前释放/写回错误 Tensor</td></tr>
    <tr><td style='text-align: left;'>ACL 初始化失败</td><td style='text-align: left;'>CANNLab 是否分配 NPU、内核与镜像是否正确</td></tr>
  </tbody>
</table>


## 8. 独立完成后核对参考答案

只有完整自检通过后再打开参考答案。下一单元默认不会展示答案。


In [ ]:
SHOW_REFERENCE_ANSWER = False
if SHOW_REFERENCE_ANSWER:
    subprocess.run(['cat', str(ANSWER)], check=True)
else:
    print('先独立完成并通过完整自检，再将开关改为 True。')


## 9. 实践小结与完成检查

完成本实践后，请确认：

- 能解释为什么 depth 固定为 1、num 改为 2；
- 能手算默认配置的 6144 Byte 每核队列有效载荷；
- <code>tileCount=1、2、7</code> 的 Host 调度模拟均通过；
- 单、双缓冲的数据覆盖与计算公式相同；
- 四类非法参数和四组 NPU 精度用例全部通过；
- 自检末尾显示 <code>SELF_CHECK PASS</code>。

至此你已经把双端队列式 Buffer 管理落实为真实 Ascend C 数据流水线，并能用调度不变量而非单次耗时判断实现是否成立。
